In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG crypto_pipeline")

DataFrame[]

In [0]:
# Databricks notebook source

# ==========================================================
# GOLD LAYER
#
# Notebook:
# 01_gold_daily_ohlc
#
# Purpose:
# Build Daily OHLC table from Silver snapshots
#
# Grain:
# One row per Coin per Day
#
# ==========================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "crypto_pipeline"

FACT_TABLE = f"{CATALOG}.silver.fact_coin_price"
GOLD_TABLE = f"{CATALOG}.gold.daily_ohlc"

spark.sql(f"USE CATALOG {CATALOG}")

# ==========================================================
# Create Gold Table
# ==========================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {GOLD_TABLE}

(

coin_sk STRING,

observation_date DATE,

open_price DOUBLE,

high_price DOUBLE,

low_price DOUBLE,

close_price DOUBLE,

average_volume DOUBLE

)

USING DELTA

""")

# ==========================================================
# Read Fact Table
# ==========================================================

fact_df = spark.table(FACT_TABLE)

# ==========================================================
# Extract Date
# ==========================================================

fact_df = (

    fact_df

    .withColumn(

        "observation_date",

        F.to_date("observation_ts")

    )

)

display(fact_df)

coin_sk,coin_id,observation_ts,current_price,market_cap,total_volume,market_cap_rank,price_change_24h,price_change_percentage_24h,observation_date
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:58:57.519Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094,2026-07-22
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,aster-2,2026-07-22T08:59:21.134Z,0.622088,1.670620127E9,6.3981992E7,46,-0.00839127419145902,-1.33094,2026-07-22
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:58:57.519Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982,2026-07-22
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,avalanche-2,2026-07-22T08:59:21.134Z,6.51,2.812036807E9,1.21168985E8,32,-0.14037606851970175,-2.10982,2026-07-22
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:58:57.519Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542,2026-07-22
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,binancecoin,2026-07-22T08:59:21.134Z,569.71,7.587212412E10,5.64901171E8,4,-7.535549706710867,-1.30542,2026-07-22
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:58:57.519Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796,2026-07-22
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,bitcoin,2026-07-22T08:59:21.134Z,65922.0,1.322409077378E12,3.2062806671E10,1,-251.1924731802137,-0.3796,2026-07-22
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:58:57.519Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842,2026-07-22
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,bitcoin-cash,2026-07-22T08:59:21.134Z,220.78,4.430395854E9,8.4996954E7,23,-2.645743369264551,-1.1842,2026-07-22


In [0]:
open_window = (

    Window

    .partitionBy(

        "coin_sk",

        "observation_date"

    )

    .orderBy(

        F.col("observation_ts").asc()

    )

)

open_df = (

    fact_df

    .withColumn(

        "rn",

        F.row_number().over(open_window)

    )

    .filter("rn=1")

    .select(

        "coin_sk",

        "observation_date",

        F.col("current_price").alias("open_price")

    )

)

display(open_df)

coin_sk,observation_date,open_price
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,2026-07-22,0.999865
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,2026-07-22,0.124072
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,2026-07-22,1.13
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,2026-07-22,1.84E-6
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,2026-07-22,1.88
2566e75a9ea09ae330ae3bb7a9417a3fcdbe5696fb322ab37f3c3d24c3004c76,2026-07-22,4106.34
2684ff29b19e1b58b61b5e02e240216e9b497f270e1c7686529bd0d1165cee2b,2026-07-22,0.057727
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,2026-07-22,0.622088
2fd9ab12e706102f24b58d4be659617f194c4dda13ee31517803460a69bd73f8,2026-07-22,1.14
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,2026-07-22,351.7


In [0]:
close_window = (

    Window

    .partitionBy(

        "coin_sk",

        "observation_date"

    )

    .orderBy(

        F.col("observation_ts").desc()

    )

)

close_df = (

    fact_df

    .withColumn(

        "rn",

        F.row_number().over(close_window)

    )

    .filter("rn=1")

    .select(

        "coin_sk",

        "observation_date",

        F.col("current_price").alias("close_price")

    )

)

display(close_df)

coin_sk,observation_date,close_price
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,2026-07-22,0.999865
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,2026-07-22,0.124072
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,2026-07-22,1.13
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,2026-07-22,1.84E-6
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,2026-07-22,1.88
2566e75a9ea09ae330ae3bb7a9417a3fcdbe5696fb322ab37f3c3d24c3004c76,2026-07-22,4106.34
2684ff29b19e1b58b61b5e02e240216e9b497f270e1c7686529bd0d1165cee2b,2026-07-22,0.057727
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,2026-07-22,0.622088
2fd9ab12e706102f24b58d4be659617f194c4dda13ee31517803460a69bd73f8,2026-07-22,1.14
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,2026-07-22,351.7


In [0]:
agg_df = (

    fact_df

    .groupBy(

        "coin_sk",

        "observation_date"

    )

    .agg(

        F.max("current_price").alias("high_price"),

        F.min("current_price").alias("low_price"),

        F.avg("total_volume").alias("average_volume")

    )

)

display(agg_df)

coin_sk,observation_date,high_price,low_price,average_volume
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,2026-07-22,0.622088,0.622088,6.3981992E7
4e2b4465ae2c4c917495b637780dff7ec2113e1f390b8aa84fbb19819b848768,2026-07-22,6.51,6.51,1.21168985E8
4be6b588bed71a3f34047d227bfad49e9f35d4a89195b7ad153f865a9450796a,2026-07-22,569.71,569.71,5.64901171E8
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,2026-07-22,65922.0,65922.0,3.2062806671E10
c4e03185c2e365e4f3bb253b58944a0252139e6cb7d6ecae7a7d2dde8ab47ef0,2026-07-22,220.78,220.78,8.4996954E7
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,2026-07-22,196.33,196.33,9.7337906E7
e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,2026-07-22,1.0,1.0,0.0
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,2026-07-22,0.124072,0.124072,8252504.0
6235bef37881100ed67526ae2a1af07ab4725e2e7b1f7ae709c29e01a6bc53e2,2026-07-22,0.172065,0.172065,2.82280937E8
b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,2026-07-22,8.6,8.6,1.7999511E8


In [0]:
gold_stage = (

    open_df

    .join(

        close_df,

        [

            "coin_sk",

            "observation_date"

        ]

    )

    .join(

        agg_df,

        [

            "coin_sk",

            "observation_date"

        ]

    )

)

display(gold_stage)

print(

    "Rows:",

    gold_stage.count()

)

coin_sk,observation_date,open_price,close_price,high_price,low_price,average_volume
124d4b609018175dd91ff6a12b372c389f9b0046298e32353f098c0b2b2995ea,2026-07-22,0.999865,0.999865,0.999865,0.999865,1.11249065E8
126bda23e48f287091cc02ebdc4a972096a6bc774223da74dc40c1ab85f18249,2026-07-22,0.124072,0.124072,0.124072,0.124072,8252504.0
15693b928f9f1c8df421b733df7d6adbcafded5444e309cecdac7a7510520edc,2026-07-22,1.13,1.13,1.13,1.13,1.35471518E9
20ef3a64f532468d1178902f57cc6e910c7d608e3c7821f8eb4450295adccaa2,2026-07-22,1.84E-6,1.84E-6,1.84E-6,1.84E-6,4.4443701E7
23fec56b98d88017bcab27610360b3ed3e1ad86e04aed9093989d62cb6090b03,2026-07-22,1.88,1.88,1.88,1.88,2.02695092E8
2566e75a9ea09ae330ae3bb7a9417a3fcdbe5696fb322ab37f3c3d24c3004c76,2026-07-22,4106.34,4106.34,4106.34,4106.34,1.2553983E8
2684ff29b19e1b58b61b5e02e240216e9b497f270e1c7686529bd0d1165cee2b,2026-07-22,0.057727,0.057727,0.057727,0.057727,7263279.0
2e718161445afbd9a5ee5c5a22835a07defc2037fe5031de1fef450dd4aac94b,2026-07-22,0.622088,0.622088,0.622088,0.622088,6.3981992E7
2fd9ab12e706102f24b58d4be659617f194c4dda13ee31517803460a69bd73f8,2026-07-22,1.14,1.14,1.14,1.14,1163410.0
314b8687941b8ffc3e7752d655b21546c41cfdcba9d04feed56d0b34852217d2,2026-07-22,351.7,351.7,351.7,351.7,1.12379732E8


Rows: 50


In [0]:
# ==========================================================
# PART 2
# Merge into Gold Daily OHLC
# ==========================================================

from delta.tables import DeltaTable

gold_delta = DeltaTable.forName(
    spark,
    GOLD_TABLE
)

(
    gold_delta.alias("target")
    .merge(
        gold_stage.alias("source"),
        """
        target.coin_sk = source.coin_sk
        AND
        target.observation_date = source.observation_date
        """
    )
    .whenMatchedUpdate(
        set={

            "open_price":"source.open_price",

            "high_price":"source.high_price",

            "low_price":"source.low_price",

            "close_price":"source.close_price",

            "average_volume":"source.average_volume"

        }
    )
    .whenNotMatchedInsert(
        values={

            "coin_sk":"source.coin_sk",

            "observation_date":"source.observation_date",

            "open_price":"source.open_price",

            "high_price":"source.high_price",

            "low_price":"source.low_price",

            "close_price":"source.close_price",

            "average_volume":"source.average_volume"

        }
    )
    .execute()
)

print("Gold Daily OHLC Merge Completed.")

Gold Daily OHLC Merge Completed.


In [0]:
gold_df = spark.table(GOLD_TABLE)

display(
    gold_df.orderBy(
        F.desc("observation_date")
    )
)

print("Rows:", gold_df.count())

coin_sk,observation_date,open_price,high_price,low_price,close_price,average_volume
e7735b4253b64a49c688af4708daa3dd47c829fd075ff74ea83fff1b99246733,2026-07-22,1.0,1.0,1.0,1.0,0.0
fc5577422cf6e3c8f1900e7e4cb0ea2f827ed90415aae720ffe1df8a04c8294f,2026-07-22,0.999967,0.999967,0.999967,0.999967,1.94128464E8
b76527ae1b44a61572d49f1ac34393c9d4b2acc91346d9bdc978b09fe711be00,2026-07-22,8.6,8.6,8.6,8.6,1.7999511E8
e44988b4e78b3b5f96854b7d376c9a1e5dddba8476196e5dd3000284a1488c5c,2026-07-22,196.33,196.33,196.33,196.33,9.7337906E7
fa30dc7298e987e69d2011d03ae7759fa16116678fbb21f7aa50a199117cbd3e,2026-07-22,3.72,3.72,3.72,3.72,1.3652852E8
89a42e9565784d167dc7ad5e50f91d45b6aa3141326e55c332043b9b322a9a3f,2026-07-22,4.23E-6,4.23E-6,4.23E-6,4.23E-6,4.3292614E7
8ef000ca6f4c3834f6470f803364ab08561720815084243c0166b902ac529b7b,2026-07-22,57.41,57.41,57.41,57.41,6.4196932E7
ae9869a81d32f512573269df55d19b5b46ed939fbe66e81b3755be68304c9406,2026-07-22,1919.35,1919.35,1919.35,1919.35,1.0336311395E10
c18cad259c9f4f8ef98ef645b2aa3dac7d726ab041d06643d5d13ffff58d358c,2026-07-22,65922.0,65922.0,65922.0,65922.0,3.2062806671E10
e2f96ad9165015037c46eab0b09fa5efb5a4ac4777053f7fa8869290b479db80,2026-07-22,0.402385,0.402385,0.402385,0.402385,1.83898876E8


Rows: 50


In [0]:
duplicates = (

    gold_df

    .groupBy(

        "coin_sk",

        "observation_date"

    )

    .count()

    .filter("count > 1")

)

display(duplicates)

print(

    "Duplicate Rows:",

    duplicates.count()

)

coin_sk,observation_date,count


Duplicate Rows: 0


In [0]:
display(

    gold_df.filter(

        (F.col("high_price") < F.col("low_price"))

        |

        (F.col("open_price").isNull())

        |

        (F.col("close_price").isNull())

    )

)

coin_sk,observation_date,open_price,high_price,low_price,close_price,average_volume
